In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import matplotlib.pyplot as plt
from ActivationExtractor import ActivationExtractor

/workspaces/gpt2small/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Check for GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float32 if device == "cpu" else torch.float16,
)
model = model.to(device)

Using device: cuda


In [3]:
# Define which layers/components to extract
layers_to_extract = ["transformer.h.0", "transformer.h.11"]

# Initialize the extractor tool. Hooks are persisted until `extractor.clear_hooks()`
# is called, so there's no option to toggle persistence per-call.
extractor = ActivationExtractor(model, tokenizer, layers_to_extract)

# Example: Extract activations
prompt = "The future of AI is"
print(f"Analyzing prompt: '{prompt}'")

results = extractor.extract(prompt)

tokens = results["tokens"]
activations = results["activations"]

print(f"Tokens: {tokens}")

for layer, act in activations.items():
    print(f"\nLayer: {layer}")
    print(f"Activation shape: {act.shape}")
    # Print first few values for the last token
    print(f"Values for last token '{tokens[-1]}': {act[0, -1, :5]}...")

# Optional: persist model state and hooks metadata to disk. Hooks metadata
# (e.g. layer names) is saved to a small JSON file; actual hook objects are
# not pickled because they are typically not serializable.
extractor.save_model_with_hooks("./saved_model", save_tokenizer=True)


Analyzing prompt: 'The future of AI is'
Tokens: ['The', 'Ġfuture', 'Ġof', 'ĠAI', 'Ġis']

Layer: transformer.h.0
Activation shape: torch.Size([1, 5, 768])
Values for last token 'Ġis': tensor([-1.7148,  0.3101, -0.2266, -0.3152,  0.7715], dtype=torch.float16)...

Layer: transformer.h.11
Activation shape: torch.Size([1, 5, 768])
Values for last token 'Ġis': tensor([ 4.4062,  9.4375,  1.6133, -2.6406,  0.9590], dtype=torch.float16)...


## Text Generation

Now let's use the `generate()` method to create new text:

In [4]:
# Generate text using the same extractor
prompt = "The future of AI is"
print(f"Generating from prompt: '{prompt}'\n")

# Generate with sampling (creative output)
result = extractor.generate(
    text=prompt,
    max_new_tokens=50,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    do_sample=True
)

print(f"Generated text:\n{result['generated_text']}")
print(f"\n{'='*60}")
print(f"Prompt tokens: {len(result['full_tokens']) - len(result['generated_tokens'])}")
print(f"Generated tokens: {len(result['generated_tokens'])}")
print(f"\nGenerated tokens: {result['generated_tokens'][:10]}...")  # Show first 10

Generating from prompt: 'The future of AI is'

Generated text:
The future of AI is a lot more murky than the present one, though. AI isn't just about having the right tools. We need to understand the way in which we interact with our machines. That's what's important, but the future of AI is a lot more

Prompt tokens: 5
Generated tokens: 50

Generated tokens: ['Ġa', 'Ġlot', 'Ġmore', 'Ġmurky', 'Ġthan', 'Ġthe', 'Ġpresent', 'Ġone', ',', 'Ġthough']...


### Generation with Activation Collection

Collect activations at each generation step to see how the model's internal state evolves:

In [5]:
# Generate text while collecting activations at each step
# Note: activations are always collected during generation
prompt = "The cat sat on the"
result = extractor.generate(
    text=prompt,
    max_new_tokens=8,
    temperature=0.7,
    do_sample=True
)

print(f"Prompt: '{prompt}'")
print(f"Generated: '{result['generated_text']}'")
print(f"\nGenerated tokens: {result['generated_tokens']}")
print(f"\nActivations collected for {len(result['activations'])} generation steps")

# Analyze activations at each generation step
for step, (token, step_activations) in enumerate(zip(result['generated_tokens'], result['activations'])):
    print(f"\nStep {step + 1} - Token: '{token}'")
    for layer_name, act in step_activations.items():
        # Get stats for the last token's activation in this step
        last_token_act = act[0, -1, :]
        print(f"  {layer_name}: shape={act.shape}, mean={last_token_act.mean():.3f}, std={last_token_act.std():.3f}")

Prompt: 'The cat sat on the'
Generated: 'The cat sat on the edge of the bed, and his hand'

Generated tokens: ['Ġedge', 'Ġof', 'Ġthe', 'Ġbed', ',', 'Ġand', 'Ġhis', 'Ġhand']

Activations collected for 8 generation steps

Step 1 - Token: 'Ġedge'
  transformer.h.0: shape=torch.Size([1, 5, 768]), mean=0.038, std=2.035
  transformer.h.11: shape=torch.Size([1, 5, 768]), mean=-0.076, std=17.141

Step 2 - Token: 'Ġof'
  transformer.h.0: shape=torch.Size([1, 6, 768]), mean=0.026, std=2.035
  transformer.h.11: shape=torch.Size([1, 6, 768]), mean=-0.177, std=15.148

Step 3 - Token: 'Ġthe'
  transformer.h.0: shape=torch.Size([1, 7, 768]), mean=0.022, std=1.954
  transformer.h.11: shape=torch.Size([1, 7, 768]), mean=-0.172, std=15.844

Step 4 - Token: 'Ġbed'
  transformer.h.0: shape=torch.Size([1, 8, 768]), mean=0.033, std=1.957
  transformer.h.11: shape=torch.Size([1, 8, 768]), mean=-0.078, std=16.109

Step 5 - Token: ','
  transformer.h.0: shape=torch.Size([1, 9, 768]), mean=0.022, std=2.051
  tr